In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import random
import matplotlib.pyplot as plt
import time


def generate_maintenance_plan(N, T, random_seed, alpha_range=(0.001, 0.05), initial_p_range=(3.2, 3.5)):
    """生成维护计划（保持不变）"""
    caissons = [i for i in range(N)]
    
    random.seed(random_seed)
    alpha_values = [round(random.uniform(*alpha_range), 6) for _ in range(N)]
    
    random.seed(random_seed+1)
    initial_p_values = [round(random.uniform(*initial_p_range), 2) for _ in range(N)]
    
    columns = ['INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE']
    data = []
    
    index = 0
    for caisson in range(N):
        current_p = initial_p_values[caisson]
        total_days = 0
        alpha = alpha_values[caisson]
        
        while total_days < T:
            time_to_3_2 = int((current_p - 3.2) / alpha)
            if total_days + time_to_3_2 >= T:
                break
            
            operation = 'x'
            data.append([index, caisson, initial_p_values[caisson], operation, total_days + time_to_3_2])
            index += 1
            total_days += time_to_3_2 + 1
            current_p = 3.5
    
    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan, alpha_values

# 🔥 🔥 🔥 从代码2复制的核心函数：计算维修周期
def calculate_maintenance_period_from_data(df, d_min_ratio=0.6, d_max_ratio=1.4):
    """从代码2复制：extract maintenance periods from data"""
    maintenance_periods = {}
    D_min_dict = {}
    D_max_dict = {}
    
    print("\n===== 从数据中提取维修周期 =====")
    print(f"{'Caisson':<10} {'间隔数':<10} {'间隔范围':<20} {'周期':<12} {'D_min':<10} {'D_max':<10}")
    print("-" * 80)
    
    for caisson_id in sorted(df['CSEM'].unique()):
        caisson_tasks = df[(df['CSEM'] == caisson_id) & (df['OPERATION'] == 'x')].sort_values('DATE')
        
        if len(caisson_tasks) < 2:
            default_period = PARAMS['window_size']
            maintenance_periods[caisson_id] = default_period
            D_min_dict[caisson_id] = int(np.ceil(default_period * d_min_ratio))
            D_max_dict[caisson_id] = int(np.ceil(default_period * d_max_ratio))
            print(f"{caisson_id:<10} ... {default_period:<12.2f} ...")  
            continue
        
        intervals = []
        for i in range(1, len(caisson_tasks)):
            interval = caisson_tasks.iloc[i]['DATE'] - caisson_tasks.iloc[i-1]['DATE']
            intervals.append(interval)
        
        maintenance_period = np.mean(intervals)
        maintenance_periods[caisson_id] = maintenance_period
        D_min_dict[caisson_id] = int(np.ceil(maintenance_period * d_min_ratio))
        D_max_dict[caisson_id] = int(np.ceil(maintenance_period * d_max_ratio))
        
        interval_min = min(intervals)
        interval_max = max(intervals)
        interval_range = f"[{interval_min}, {interval_max}]"
        
        print(f"{caisson_id:<10} {len(intervals):<10} {interval_range:<20} "
              f"{maintenance_period:<12.2f} {D_min_dict[caisson_id]:<10} {D_max_dict[caisson_id]:<10}")
    
    print("===== 提取完成 =====\n")
    return maintenance_periods, D_min_dict, D_max_dict

# 🔥 🔥 🔥 从代码2复制的核心聚类函数（完全替换）
def clustering_CR(df_window, window_size, window_start, global_task_count=None, full_df=None, D_min_dict=None, D_max_dict=None):
    """代码2的Gurobi版本 - 完全替换原函数"""
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window

    N = df_window.shape[0]
    mdl = gp.Model('Dynamic_Window_Clustering')
    mdl.setParam('OutputFlag', 0)

    data_pos_num = window_size + 1
    theta = mdl.addVars(N, data_pos_num, vtype=GRB.BINARY, name='theta')
    CS = mdl.addVars(data_pos_num, lb=0.0, vtype=GRB.CONTINUOUS, name='CS')
    y = mdl.addVars(data_pos_num, vtype=GRB.BINARY, name='y')

    # 任务分配约束
    for n in range(N):
        mdl.addConstr(theta.sum(n, '*') == 1, name=f"assign_task_{n}")

    # 容量约束
    for t in range(data_pos_num):
        mdl.addConstr(theta.sum('*', t) <= PARAMS['Cmax'], name=f"cap_{t}")

    # 时间窗口约束
    a = [0] * N
    b = [0] * N
    for n in range(N):
        local_time = int(df_window.iloc[n]['window_time'])
        a[n] = int(max(0, local_time - PARAMS['advanced']))
        b[n] = int(min(data_pos_num, local_time + int((3.2 - 3.0)/alpha_values[int(df_window.iloc[n]["CSEM"])])//2))

    for n in range(N):
        for t in range(data_pos_num):
            mdl.addConstr(t * theta[n, t] + a[n] * (1 - theta[n, t]) >= a[n], name=f"lb_range_n{n}_t{t}")
            mdl.addConstr(t * theta[n, t] + b[n] * (1 - theta[n, t]) <= b[n], name=f"ub_range_n{n}_t{t}")

    # 聚类激活约束
    M = PARAMS['Cmax']
    for t in range(data_pos_num):
        task_count = theta.sum('*', t)
        mdl.addConstr(task_count + M * (1 - y[t]) >= 1, name=f"activate_lb_{t}")
        mdl.addConstr(task_count <= M * y[t], name=f"activate_ub_{t}")

    # 🔥 关键差异1：新的目标函数（最大化利润 = setup_cost - CP_total）
    setup_cost = PARAMS['Cs'] * (theta.sum() - y.sum())
    
    # CP成本矩阵
    cost_matrix = {}
    for n in range(N):
        local_time_n = int(df_window.iloc[n]['window_time'])
        for t in range(data_pos_num):
            deviation = t - local_time_n
            if deviation > 0:
                cost_matrix[(n, t)] = PARAMS['Cbeta'] * deviation
            elif deviation < 0:
                cost_matrix[(n, t)] = PARAMS['Calpha'] * (-deviation)
            else:
                cost_matrix[(n, t)] = 0
    
    CP_total = gp.quicksum(cost_matrix[(n, t)] * theta[n, t] for n in range(N) for t in range(data_pos_num))
    obj = setup_cost - CP_total  # 🔥 最大化利润
    mdl.setObjective(obj, GRB.MAXIMIZE)

    # 🔥 关键差异2：新增维护间隔约束
    if window_start > 0 and full_df is not None and D_min_dict is not None:
        for n in range(N):
            row = df_window.iloc[n]
            caisson = int(row['CSEM'])
            
            prev_rows = full_df[
                (full_df['CSEM'] == caisson) &
                (full_df['CENTER'] < row['DATE']) &
                (full_df['OPERATION'] == 'x')
            ].sort_values('CENTER')
            
            if not prev_rows.empty:
                prev_center = int(prev_rows.iloc[-1]['CENTER'])
                t_now_local = gp.quicksum(t * theta[n, t] for t in range(data_pos_num))
                d_min_i = D_min_dict.get(caisson, PARAMS['window_size'] * 0.6)
                d_max_i = D_max_dict.get(caisson, PARAMS['window_size'] * 1.4)
                
                mdl.addConstr(t_now_local + window_start - prev_center >= d_min_i, name=f"dmin_n{n}")
                mdl.addConstr(t_now_local + window_start - prev_center <= d_max_i, name=f"dmax_n{n}")

    # 求解参数
    mdl.setParam('LogToConsole', 1)
    mdl.setParam('OutputFlag', 1)
    mdl.setParam('TimeLimit', 300)
    mdl.setParam('MIPGap', 0.05)

    print(f"Window {window_start}: N_tasks={N}")
    mdl.optimize()

    if mdl.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print(f"Failed to solve window {window_start}: {mdl.status}")
        return None
    
    # 结果提取
    centers = {}
    for n in range(N):
        for t in range(data_pos_num):
            val = theta[n, t].x
            if val is not None and round(val) == 1:
                centers[n] = t + window_start
                break

    index_n_list = df_window.INDEX.tolist()
    for n in range(len(index_n_list)):
        df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

    print(f"Model stats: {mdl.NumVars} vars, {mdl.NumConstrs} cons")
    print(f"Objective value: {mdl.ObjVal}")
    return df_window

# 🔥 保持原有的泄漏计算函数不变
def calculate_gas_leakage(df, T, use_center=True):
    total_leakage = 0
    for csem in df['CSEM'].unique():
        csem_data = df[df['CSEM'] == csem]
        x_operations = csem_data[csem_data['OPERATION'] == 'x'].sort_values('DATE' if not use_center else 'CENTER')
        if x_operations.empty:
            continue
            
        alpha = alpha_values[int(csem)]

        # 任务间泄漏
        for i in range(1, len(x_operations)):
            current_time = x_operations.iloc[i][('CENTER' if use_center else 'DATE')]
            prev_time = x_operations.iloc[i-1][('CENTER' if use_center else 'DATE')]
            center_diff = current_time - prev_time - 1
            total_leakage += max(0, center_diff) * alpha * PARAMS['K']

        # 开头泄漏
        first_time = x_operations.iloc[0][('CENTER' if use_center else 'DATE')]
        total_leakage += first_time * alpha * PARAMS['K']

        # 结尾泄漏
        last_time = x_operations.iloc[-1][('CENTER' if use_center else 'DATE')]
        leak_time = max(0, T - (last_time + 1))
        total_leakage += leak_time * alpha * PARAMS['K']
    return total_leakage

def update_secondary_tasks(main_df, clustered_df, window_start, window_end):
    """Update the secondary tasks time (使用代码2的逻辑）"""
    for caisson in clustered_df["CSEM"].unique():
        # 当前 caisson 在当前窗口内的 x 任务
        tasks = main_df[
            (main_df['CSEM'] == caisson) & 
            (main_df['DATE'].between(window_start, window_end)) & 
            (main_df['OPERATION'] == 'x')
        ].sort_values('DATE')
        
        if len(tasks) > 1:
            # 第二个任务的 index（保持代码2逻辑）
            second_idx = tasks.index[1]
            
            # 该 caisson 在本窗口聚类后的中心时间
            center_time = clustered_df.loc[clustered_df['CSEM'] == caisson, 'CENTER'].iloc[0]
            
            # 按代码2方式计算新的日期
            alpha = alpha_values[int(caisson)]
            new_date = int(center_time + 1 + (3.5 - 3.2) / alpha)
            
            # 同时更新 DATE 和 CENTER
            main_df.loc[second_idx, ['DATE', 'CENTER']] = new_date, new_date


# 🔥 主函数修改：集成代码2逻辑
def run_clustering_analysis(params, random_seed):
    """完整聚类分析函数 - 修改版"""
    global alpha_values, PARAMS  # 确保全局变量同步
    PARAMS.update(params)
    
    # 生成维护计划
    df, alpha_values_list = generate_maintenance_plan(
        N=params['P'], T=params['T'], random_seed=random_seed
    )
    alpha_values = {i: alpha_values_list[i] for i in range(params['P'])}
    
    print(f"Generated {len(df)} tasks for seed {random_seed}")
    
    # 🔥 新增：计算维修周期约束（代码2特性）
    maintenance_periods, D_min_dict, D_max_dict = calculate_maintenance_period_from_data(df)
    
    # CPU时间测量
    clustering_cpu_start = time.process_time()
    
    df_init = df.copy()
    df_history_list = [df_init]
    sliding_windows_starts = []
    sliding_windows_ends = []
    df['CENTER'] = df['DATE']
    
    window_start = 0
    window_size = params['window_size']
    
    while window_start <= params['T']:
        window_end = min(window_start + window_size, params['T'])
        sliding_windows_starts.append(window_start)
        sliding_windows_ends.append(window_end)
        
        mask = (df['DATE'] >= window_start) & (df['DATE'] <= window_end) & (df['OPERATION']== 'x')
        window_df = df.loc[mask].copy()
        if window_df.empty:
            break
            
        window_df['window_time'] = window_df['DATE'] - window_start
        first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])

        # 🔥 使用新的clustering_CR（带维护约束）
        clustered_df = clustering_CR(
            first_tasks, window_size, window_start, 
            full_df=df, D_min_dict=D_min_dict, D_max_dict=D_max_dict
        )
        
        if clustered_df is not None:
            center_map = clustered_df.set_index('INDEX')['CENTER']
            df.loc[df['INDEX'].isin(center_map.index), 'CENTER'] = df['INDEX'].map(center_map)
            update_secondary_tasks(df, clustered_df, window_start, window_end)
        
        # 滑动窗口逻辑（保持不变）
        second_dates = []
        if clustered_df is not None:
            for caisson in clustered_df["CSEM"].unique():
                tasks = df[(df['CSEM'] == caisson) & 
                          (df['OPERATION'] == 'x') &
                          (df['DATE'].between(window_start, window_end))].sort_values('DATE')
                if len(tasks) > 1:
                    second_dates.append(tasks.iloc[1]['DATE'])
        
        window_start = min(second_dates) if second_dates else window_start + window_size
        df_history_list.append(df.copy())
        
        if window_end >= params['T']:
            break
    
    # 处理未分配任务
    mask_remain = (df['OPERATION'] == "x") & (df['CENTER'] == df['DATE'])
    df.loc[mask_remain, 'CENTER'] = df.loc[mask_remain, 'DATE']

    clustering_cpu_end = time.process_time()
    total_clustering_cpu_time = clustering_cpu_end - clustering_cpu_start

    # 计算泄漏（保持不变）
    leakage_before = calculate_gas_leakage(df_init, params['T'], use_center=False)
    leakage_after = calculate_gas_leakage(df, params['T'], use_center=True)
    reduction = leakage_before - leakage_after
    reduction_pct = (reduction / leakage_before * 100) if leakage_before > 0 else 0
    
    return {
        'final_df': df,
        'df_history_list': df_history_list,
        'sliding_windows_starts': sliding_windows_starts,
        'sliding_windows_ends': sliding_windows_ends,
        'alpha_values': alpha_values_list,
        'maintenance_plan': df_init,
        'D_min_dict': D_min_dict,
        'D_max_dict': D_max_dict,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'reduction': reduction,
        'reduction_pct': reduction_pct,
        'total_clustering_cpu_time': total_clustering_cpu_time
    }

def calculate_detailed_stats(result_dict, params):
    """
    计算所有详细指标，用于CSV输出
    """
    df_orig = result_dict['maintenance_plan'].copy()
    result_df = result_dict['final_df'].copy()
    
    # 只针对x任务，确保INDEX对齐
    df_x = df_orig[df_orig['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    res_x = result_df[result_df['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    
    # CP成本计算
    time_diffs = res_x['CENTER'].values - df_x['DATE'].values
    cp_cost = np.sum(
        np.where(time_diffs < 0, np.abs(time_diffs) * params['Calpha'], time_diffs * params['Cbeta'])
    )
    
    # 聚类统计
    center_counts = res_x['CENTER'].value_counts()
    num_groups = len(center_counts)
    avg_tasks_per_group = center_counts.mean()
    
    # 时间偏移统计
    num_tasks = len(time_diffs)
    avg_shift_abs = np.mean(np.abs(time_diffs))
    max_shift_abs = np.max(np.abs(time_diffs))
    
    # 提前统计
    advanced_mask = time_diffs < 0
    avg_advance = np.mean(np.abs(time_diffs[advanced_mask])) if np.any(advanced_mask) else 0
    num_advanced = np.sum(advanced_mask)
    pct_advanced = num_advanced / num_tasks * params['Cs']
    
    # 延迟统计
    delayed_mask = time_diffs > 0
    avg_delay = np.mean(time_diffs[delayed_mask]) if np.any(delayed_mask) else 0
    num_delayed = np.sum(delayed_mask)
    pct_delayed = num_delayed / num_tasks * params['Cs']
    
    # 不变统计
    num_unchanged = np.sum(time_diffs == 0)
    pct_unchanged = num_unchanged / num_tasks * params['Cs']
    
    # 成本统计
    operation_x_count = len(df_x)
    cs_cost_before = operation_x_count * params['Cs']
    cs_cost_after = num_groups * params['Cs']
    cs_cost_reduction = cs_cost_before - cs_cost_after
    cs_cost_reduction_pct = cs_cost_reduction / cs_cost_before * 100
    total_cost = cs_cost_after + cp_cost
    
    # 泄漏统计（来自result_dict）
    leakage_before = result_dict['leakage_before']
    leakage_after = result_dict['leakage_after']
    leakage_reduction = result_dict['reduction']
    leakage_reduction_pct = result_dict['reduction_pct']

    # 🔥 整个聚类过程CPU时间
    total_clustering_cpu_time = result_dict.get('total_clustering_cpu_time', 0)
    
    # 完整统计字典（匹配你的表格列）
    stats = {
        # 基本信息
        'seed': None,  # 稍后填充
        'nb_groups': num_groups,
        'Avg_tasks/group': avg_tasks_per_group,
        
        # CP成本
        'CP_cost': cp_cost,
        
        # 时间偏移
        'Avg_shift_(abs)': avg_shift_abs,
        'Max_shift_(abs)': max_shift_abs,
        'Avg_advance_(days)': avg_advance,
        'Avg_delay_(days)': avg_delay,
        'pct_advanced_%': pct_advanced,
        'pct_delayed_%': pct_delayed,
        'pct_unchanged_%': pct_unchanged,
        'num_advanced': num_advanced,
        'num_delayed': num_delayed,
        'num_unchanged': num_unchanged,
        'total_tasks': num_tasks,
        
        # CS成本
        'CS_cost_before': cs_cost_before,
        'CS_cost_after': cs_cost_after,
        # 'CS_cost_reduction': cs_cost_reduction,
        # 'CS_cost_reduction_pct': cs_cost_reduction_pct,

        # Total成本
        'Total_cost': total_cost,
        
        # 泄漏统计
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'leakage_reduction': leakage_reduction,
        'leakage_reduction_pct': leakage_reduction_pct,

        # CPU时间
        'total_clustering_cpu_s': total_clustering_cpu_time,
    }
    
    return stats


def run_analysis_with_csv_export(params, seeds, output_filename='detailed_results.csv'):
    """
    运行多种子分析并导出完整CSV（包含所有表格列）
    """
    all_results = []
    
    for seed in seeds:
        print(f"\n🔄 Processing seed {seed}...")
        
        # 运行分析
        result_dict = run_clustering_analysis(params, random_seed=seed)
        
        # 计算详细统计
        stats = calculate_detailed_stats(result_dict, params)
        stats['seed'] = seed  # 填充seed信息
        
        all_results.append(stats)
        
        # 打印摘要
        print(f"✅ Seed {seed}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")
    
    # 转换为DataFrame并排序列顺序（匹配你的表格）
    results_df = pd.DataFrame(all_results)
    
    # 按你的截图调整列顺序
    column_order = [
        'seed', 'nb_groups', 'Avg_tasks/group',  'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost',
        'Avg_shift_(abs)', 'Max_shift_(abs)', 'Avg_advance_(days)', 'Avg_delay_(days)','pct_advanced_%', 
        'pct_delayed_%', 'pct_unchanged_%', 'leakage_before', 'leakage_after', 'leakage_reduction', 
        'leakage_reduction_pct', 'total_clustering_cpu_s'
    ]
    
    # 只保留存在的列并排序
    available_columns = [col for col in column_order if col in results_df.columns]
    results_df = results_df[available_columns]
    
    # 保存CSV
    results_df.to_csv(output_filename, index=False, float_format='%.3f')
    print(f"\n📁 完整结果已保存到: {output_filename}")
    print(f"📊 包含 {len(results_df)} 种子, {len(results_df.columns)} 个指标列")
    
    return results_df


def visualize_clustering_results(result_dict, params, seed_name="default", save_csv=True, show_plot=True):
    """可视化聚类结果"""
    result_df = result_dict['final_df']
    maintenance_plan = result_dict['maintenance_plan']
    
    # 成本计算
    operation_x_count = (result_df['OPERATION'] == 'x').sum()
    before_cost = operation_x_count * params['Cs'] 
    operation_x_center_unique_count = result_df[result_df['OPERATION'] == 'x']['CENTER'].nunique()
    after_cost = operation_x_center_unique_count * params['Cs'] 
    
    print(f"Cost before grouping: {before_cost}")
    print(f"Cost after grouping: {after_cost}")
    
    # 泄漏统计
    leakage_before = round(result_dict['leakage_before'], 3)
    leakage_after = round(result_dict['leakage_after'], 3)
    print(f"Gas leakage before: {leakage_before}")
    print(f"Gas leakage after: {leakage_after}")
    
    # 节约统计
    cost_reduction = before_cost - after_cost
    cost_reduction_pct = (cost_reduction / before_cost * 100) if before_cost > 0 else 0
    print(f"Cost reduction: {cost_reduction} ({cost_reduction_pct:.1f}%)")
    print(f"Gas leakage reduction: {result_dict['reduction']:.3f} ({result_dict['reduction_pct']:.1f}%)")
    
    # 计算统计指标（修复UnboundLocalError）
    result_df_x_count = len(result_df[result_df['OPERATION'] == 'x'])
    avg_tasks_per_cluster = result_df_x_count / operation_x_center_unique_count if operation_x_center_unique_count > 0 else 0
    
    if show_plot:
        plt.figure(figsize=(8, 7))
        
        df_original_x = maintenance_plan[maintenance_plan['OPERATION'] == 'x']
        result_df_x = result_df[result_df['OPERATION'] == 'x']
        
        df_x = df_original_x.set_index('INDEX')
        result_x = result_df_x.set_index('INDEX')
        common_indices = df_x.index.intersection(result_x.index)
        df_x = df_x.loc[common_indices].reset_index()
        result_x = result_x.loc[common_indices].reset_index()
        
        time_shift = result_x['CENTER'] - df_x['DATE']
        plt.scatter(df_x['DATE'], result_x['CENTER'], 
                   c=time_shift, cmap='RdBu_r', 
                   vmin=-max(abs(time_shift.min()), abs(time_shift.max())), 
                   vmax=max(abs(time_shift.min()), abs(time_shift.max())), 
                   alpha=0.7, s=50, edgecolors='black', linewidth=0.8)
        
        plt.colorbar(label='Time Shift (days)\nRed=delayed, Blue=advanced')
        plt.title(f"Cluster Center Distribution (Seed {seed_name})", fontsize=16, fontweight='bold')
        plt.xlabel("Original time", fontsize=14)
        plt.ylabel("Cluster center", fontsize=14)
        plt.grid(True, alpha=0.3)
        
        max_range = max(result_x['DATE'].max(), result_x['CENTER'].max() + 5)
        plt.xlim(0, max_range)
        plt.ylim(0, max_range)
        plt.plot([0, max_range], [0, max_range], color='gray', linestyle='--', linewidth=2, label='45° diagonal')
        plt.legend()
        plt.gca().set_aspect('equal')
        plt.tight_layout()
        plt.show()
        
        center_counts = result_df_x['CENTER'].value_counts().sort_index()
        print("\nCluster center usage:")
        print(center_counts)
        print(f"Total clusters used: {len(center_counts)}")
        print(f"Average tasks per cluster: {avg_tasks_per_cluster:.1f}")
    
    if save_csv:
        filename = f'optimized_plan_seed_{seed_name}.csv'
        result_df.to_csv(filename, index=False)
        print(f"\nResult saved to '{filename}'")
    
    print(f"\nSummary:")
    print(f"- Cost saved: {cost_reduction_pct:.1f}%")
    print(f"- Gas leakage reduced: {result_dict['reduction_pct']:.1f}%")
    print(f"- Clusters created: {operation_x_center_unique_count}")


    return {
        'before_cost': before_cost,
        'after_cost': after_cost,
        'cost_reduction': cost_reduction,
        'cost_reduction_pct': cost_reduction_pct,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'clusters_used': operation_x_center_unique_count,
        'avg_tasks_per_cluster': avg_tasks_per_cluster
    }

# # 使用示例 - 生成你的表格CSV if we fixed P and iterate seeds
# PARAMS = {
#     'P': 100, 
#     'T': 180, 
#     'Cmax': 20, 
#     'window_size': 10, 
#     'advanced': 3, 
#     'K': 5.993, 
#     'Calpha': 5, 
#     'Cbeta': 10, 
#     'Cs': 100}

# seeds_to_test = [40]

# # name the file if sensitivity for P
# detailed_results_df = run_analysis_with_csv_export(
#     PARAMS, seeds_to_test, 
#     output_filename=f'Sensitivity_N/cost{PARAMS["P"]}.csv'
# )

# # 显示前几行
# print("\n📋 CSV预览:")
# print(detailed_results_df.round(3).head())
# visualize_clustering_results(
#     run_clustering_analysis(PARAMS, random_seed=40),
#     PARAMS, seed_name=40, save_csv=False, show_plot=True)



# 🔥 新实验：固定seed=1，P从10-200循环
# seed = 6  # 固定单一seed
# P_values = list(range(10, 201, 10))  # 10,20,...,200，共19个值

# base_params = {
#     'T': 180, 
#     'Cmax': 20, 
#     'window_size': 10, 
#     'advanced': 3, 
#     'K': 5.993, 
#     'Calpha': 5, 
#     'Cbeta': 10, 
#     'Cs': 100
# }

# all_results = []

# for p in P_values:
#     print(f"\n🔄 Running P={p} with seed={seed}...")
    
#     params = base_params.copy()
#     params['P'] = p  # 更新P
    
#     # 运行分析（单seed）
#     result_dict = run_clustering_analysis(params, random_seed=seed)
#     stats = calculate_detailed_stats(result_dict, params)
#     stats['seed'] = seed
#     stats['P'] = p  # 添加P列
    
#     all_results.append(stats)
    
#     print(f"✅ P={p}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")

# # DataFrame并保存
# results_df = pd.DataFrame(all_results)

# # 🔥 你以前的精确列顺序（从CSV头）
# original_column_order = [
#     'seed', 'nb_groups', 'Avg_tasks/group', 
#     'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 
#     'Avg_shift_(abs)', 'Max_shift_(abs)', 
#     'Avg_advance_(days)', 'Avg_delay_(days)', 
#     'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 
#     'leakage_before', 'leakage_after', 'leakage_reduction', 'leakage_reduction_pct', 
#     'total_clustering_cpu_s'
# ]

# # 🔥 P放最前，其余严格按原顺序
# column_order = ['P'] + original_column_order
# available_columns = [col for col in column_order if col in results_df.columns]
# results_df = results_df[available_columns]

# filename = f'Sensitivity_P10-200_seed{seed}_all_metrics.csv'
# results_df.to_csv(filename, index=False, float_format='%.3f')
# print(f"\n📁 结果保存到: {filename} (共{len(results_df)}行，{len(results_df.columns)}列)")

# # 显示预览
# print("\n📋 结果预览:")
# print(results_df.round(3)[['P', 'nb_groups', 'CP_cost', 'Total_cost', 'total_clustering_cpu_s']].head(10))

# # 可视化最后一个P的结果
# visualize_clustering_results(
#     run_clustering_analysis({**base_params, 'P': 200}, random_seed=seed),
#     {**base_params, 'P': 200}, seed_name=f"{seed}_P200", save_csv=True)




# # 🔥 新实验：固定seed=1，P从10-200循环
# for seed in range(7, 31):  
#     P_values = list(range(10, 201, 10))  # 10,20,...,200，共19个值

#     base_params = {
#         'T': 180, 
#         'Cmax': 20, 
#         'window_size': 10, 
#         'advanced': 3, 
#         'K': 5.993, 
#         'Calpha': 5, 
#         'Cbeta': 10, 
#         'Cs': 100
#     }

#     all_results = []

#     for p in P_values:
#         print(f"\n🔄 Running P={p} with seed={seed}...")
        
#         params = base_params.copy()
#         params['P'] = p  # 更新P
        
#         # 运行分析（单seed）
#         result_dict = run_clustering_analysis(params, random_seed=seed)
#         stats = calculate_detailed_stats(result_dict, params)
#         stats['seed'] = seed
#         stats['P'] = p  # 添加P列
        
#         all_results.append(stats)
        
#         print(f"✅ P={p}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")

#     # DataFrame并保存
#     results_df = pd.DataFrame(all_results)

#     # 🔥 你以前的精确列顺序（从CSV头）
#     original_column_order = [
#         'seed', 'nb_groups', 'Avg_tasks/group', 
#         'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 
#         'Avg_shift_(abs)', 'Max_shift_(abs)', 
#         'Avg_advance_(days)', 'Avg_delay_(days)', 
#         'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 
#         'leakage_before', 'leakage_after', 'leakage_reduction', 'leakage_reduction_pct', 
#         'total_clustering_cpu_s'
#     ]

#     # 🔥 P放最前，其余严格按原顺序
#     column_order = ['P'] + original_column_order
#     available_columns = [col for col in column_order if col in results_df.columns]
#     results_df = results_df[available_columns]

#     filename = f'Sensitivity_P10-200_seed{seed}_all_metrics.csv'
#     results_df.to_csv(filename, index=False, float_format='%.3f')
#     print(f"\n📁 结果保存到: {filename} (共{len(results_df)}行，{len(results_df.columns)}列)")

#     # 显示预览
#     print("\n📋 结果预览:")
#     print(results_df.round(3)[['P', 'nb_groups', 'CP_cost', 'Total_cost', 'total_clustering_cpu_s']].head(10))

#     # 可视化最后一个P的结果
#     visualize_clustering_results(
#         run_clustering_analysis({**base_params, 'P': 200}, random_seed=seed),
#         {**base_params, 'P': 200}, seed_name=f"{seed}_P200", save_csv=True)



# # 🔥 新实验：固定seed=1，T从10-200循环
# for seed in range(1, 31):
#     T_values = list(range(30, 361, 30))  # 30,60,...,360，共12个值

#     base_params = {
#         'P': 100, 
#         'Cmax': 20, 
#         'window_size': 10, 
#         'advanced': 3, 
#         'K': 5.993, 
#         'Calpha': 5, 
#         'Cbeta': 10, 
#         'Cs': 100
#     }

#     all_results = []

#     for t in T_values:
#         print(f"\n🔄 Running T={t} with seed={seed}...")
        
#         params = base_params.copy()
#         params['T'] = t  # 更新T
        
#         # 运行分析（单seed）
#         result_dict = run_clustering_analysis(params, random_seed=seed)
#         stats = calculate_detailed_stats(result_dict, params)
#         stats['seed'] = seed
#         stats['T'] = t  # 添加T列
        
#         all_results.append(stats)
        
#         print(f"✅ T={t}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")

#     # DataFrame并保存
#     results_df = pd.DataFrame(all_results)

#     # 🔥 你以前的精确列顺序（从CSV头）
#     original_column_order = [
#         'seed', 'nb_groups', 'Avg_tasks/group', 
#         'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 
#         'Avg_shift_(abs)', 'Max_shift_(abs)', 
#         'Avg_advance_(days)', 'Avg_delay_(days)', 
#         'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 
#         'leakage_before', 'leakage_after', 'leakage_reduction', 'leakage_reduction_pct', 
#         'total_clustering_cpu_s'
#     ]

#     # 🔥 P放最前，其余严格按原顺序
#     column_order = ['T'] + original_column_order
#     available_columns = [col for col in column_order if col in results_df.columns]
#     results_df = results_df[available_columns]

#     filename = f'Sensitivity_T30-360_seed{seed}_all_metrics.csv'
#     results_df.to_csv(filename, index=False, float_format='%.3f')
#     print(f"\n📁 结果保存到: {filename} (共{len(results_df)}行，{len(results_df.columns)}列)")

#     # 显示预览
#     print("\n📋 结果预览:")
#     print(results_df.round(3)[['T', 'nb_groups', 'CP_cost', 'Total_cost', 'total_clustering_cpu_s']].head(10))

#     # 可视化最后一个P的结果
#     visualize_clustering_results(
#         run_clustering_analysis({**base_params, 'T': 360}, random_seed=seed),
#         {**base_params, 'T': 360}, seed_name=f"{seed}_T360", save_csv=True)




# import time
# import threading
# import pandas as pd  # 确保导入

# # 🔥 配置（你的原始参数）
# P_values = list(range(10, 201, 10))
# base_params = {
#     'T': 180, 
#     'Cmax': 20, 
#     'window_size': 10, 
#     'advanced': 3, 
#     'K': 5.993, 
#     'Calpha': 5, 
#     'Cbeta': 10, 
#     'Cs': 100
# }

# TIMEOUT_SECONDS_PER_SEED = 180  # 3分钟
# MAX_ATTEMPTS = 50
# TARGET_SEEDS = 30

# class SeedTimeoutException(Exception):
#     pass

# def seed_timeout_monitor(stop_event):
#     time.sleep(TIMEOUT_SECONDS_PER_SEED)
#     stop_event.set()
#     raise SeedTimeoutException("Seed超时3分钟")

# # 🔥 初始化变量
# successful_count = 0

# print(f"🚀 Seed超时保护版 - 目标{ TARGET_SEEDS }个完整结果")

# # 🔥 你的原始循环结构 + 超时保护
# for seed_attempt in range(MAX_ATTEMPTS):
#     if successful_count >= TARGET_SEEDS:
#         break
        
#     seed = seed_attempt + 1
#     print(f"\n🔄 Processing seed {seed}... (总超时3min)")
    
#     # 🔥 Seed超时监控
#     stop_event = threading.Event()
#     timeout_thread = threading.Thread(target=seed_timeout_monitor, args=(stop_event,))
#     timeout_thread.daemon = True
#     timeout_thread.start()
    
#     all_results = []
#     seed_success = True
    
#     try:
#         for p in P_values:
#             if stop_event.is_set():
#                 print(f"\n⏰ Seed={seed} 总超时3分钟，换下一个seed！")
#                 seed_success = False
#                 break
            
#             print(f"\n🔄 Running P={p} with seed={seed}...")
            
#             params = base_params.copy()
#             params['P'] = p
            
#             # 🔥 无P超时，直接运行你的函数
#             result_dict = run_clustering_analysis(params, random_seed=seed)
#             stats = calculate_detailed_stats(result_dict, params)
#             stats['seed'] = seed
#             stats['P'] = p
#             all_results.append(stats)
            
#             print(f"✅ P={p}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")
        
#         # 🔥 停止超时线程
#         stop_event.set()
#         timeout_thread.join(timeout=1)
        
#         if seed_success and len(all_results) == len(P_values):
#             # 🔥 🔥 🔥 完全你的原始保存逻辑！
#             results_df = pd.DataFrame(all_results)
            
#             # 🔥 你以前的精确列顺序
#             original_column_order = [
#                 'seed', 'nb_groups', 'Avg_tasks/group', 
#                 'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 
#                 'Avg_shift_(abs)', 'Max_shift_(abs)', 
#                 'Avg_advance_(days)', 'Avg_delay_(days)', 
#                 'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 
#                 'leakage_before', 'leakage_after', 'leakage_reduction', 
#                 'leakage_reduction_pct', 'total_clustering_cpu_s'
#             ]
            
#             # 🔥 P放最前，完全一样！
#             column_order = ['P'] + original_column_order
#             available_columns = [col for col in column_order if col in results_df.columns]
#             results_df = results_df[available_columns]
            
#             # 🔥 🔥 完全一样的文件名！
#             filename = f'Sensitivity_P_COST/Sensitivity_P10-200_seed{seed}_COST.csv'
#             results_df.to_csv(filename, index=False, float_format='%.3f')
#             print(f"\n📁 结果保存到: {filename} (共{len(results_df)}行，{len(results_df.columns)}列)")
            
#             # 🔥 你的预览，完全一样！
#             print("\n📋 结果预览:")
#             print(results_df.round(3)[['P', 'nb_groups', 'CP_cost', 'Total_cost', 'total_clustering_cpu_s']].head(10))
            
#             # # 🔥 你的可视化，完全一样！
#             # visualize_clustering_results(
#             #     run_clustering_analysis({**base_params, 'P': 200}, random_seed=seed),
#             #     {**base_params, 'P': 200}, seed_name=f"{seed}_P200", save_csv=True
#             # )
            
#             successful_count += 1
#             print(f"✅ Seed={seed} 完美！进度：{successful_count}/{TARGET_SEEDS}")
#         else:
#             print(f"❌ Seed={seed} 超时中断，尝试下一个...")
            
#     except Exception as e:
#         print(f"💥 Seed={seed} 异常：{str(e)}")
#         stop_event.set()
#         timeout_thread.join(timeout=1)

# print(f"\n🎉 批量实验完成！成功{successful_count}/{TARGET_SEEDS}个seed")



import time
import threading
import pandas as pd  # 确保导入


# 🔥 配置（修改：固定P=100，T从30-360每隔30）
T_values = list(range(30, 361, 30))  # 30,60,90,...,360
base_params = {
    'P': 100,  # 固定P=100
    'Cmax': 20, 
    'window_size': 10, 
    'advanced': 3, 
    'K': 5.993, 
    'Calpha': 5, 
    'Cbeta': 10, 
    'Cs': 100
}


TIMEOUT_SECONDS_PER_SEED = 180  # 3分钟
MAX_ATTEMPTS = 50
TARGET_SEEDS = 30


class SeedTimeoutException(Exception):
    pass


def seed_timeout_monitor(stop_event):
    time.sleep(TIMEOUT_SECONDS_PER_SEED)
    stop_event.set()
    raise SeedTimeoutException("Seed超时3分钟")


# 🔥 初始化变量
successful_count = 0


print(f"🚀 Seed超时保护版 - 固定P=100，T=30-360 - 目标{ TARGET_SEEDS }个完整结果")


# 🔥 你的原始循环结构 + 超时保护
for seed_attempt in range(MAX_ATTEMPTS):
    if successful_count >= TARGET_SEEDS:
        break
        
    seed = seed_attempt + 1
    print(f"\n🔄 Processing seed {seed}... (总超时3min)")
    
    # 🔥 Seed超时监控
    stop_event = threading.Event()
    timeout_thread = threading.Thread(target=seed_timeout_monitor, args=(stop_event,))
    timeout_thread.daemon = True
    timeout_thread.start()
    
    all_results = []
    seed_success = True
    
    try:
        for t in T_values:
            if stop_event.is_set():
                print(f"\n⏰ Seed={seed} 总超时3分钟，换下一个seed！")
                seed_success = False
                break
            
            print(f"\n🔄 Running T={t} with seed={seed}...")
            
            params = base_params.copy()
            params['T'] = t
            
            # 🔥 无T超时，直接运行你的函数
            result_dict = run_clustering_analysis(params, random_seed=seed)
            stats = calculate_detailed_stats(result_dict, params)
            stats['seed'] = seed
            stats['T'] = t  # 修改为T
            all_results.append(stats)
            
            print(f"✅ T={t}: CP_cost={stats['CP_cost']:.1f}, Groups={stats['nb_groups']}")
        
        # 🔥 停止超时线程
        stop_event.set()
        timeout_thread.join(timeout=1)
        
        if seed_success and len(all_results) == len(T_values):
            # 🔥 🔥 🔥 完全你的原始保存逻辑！
            results_df = pd.DataFrame(all_results)
            
            # 🔥 你以前的精确列顺序（添加T）
            original_column_order = [
                'seed', 'nb_groups', 'Avg_tasks/group', 
                'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 
                'Avg_shift_(abs)', 'Max_shift_(abs)', 
                'Avg_advance_(days)', 'Avg_delay_(days)', 
                'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 
                'leakage_before', 'leakage_after', 'leakage_reduction', 
                'leakage_reduction_pct', 'total_clustering_cpu_s'
            ]
            
            # 🔥 T放最前，完全一样！
            column_order = ['T'] + original_column_order  # 修改为T
            available_columns = [col for col in column_order if col in results_df.columns]
            results_df = results_df[available_columns]
            
            # 🔥 🔥 修改文件名：Sensitivity_T_MIMAR
            filename = f'Sensitivity_T_COST/Sensitivity_T30-360_seed{seed}_COST.csv'
            results_df.to_csv(filename, index=False, float_format='%.3f')
            print(f"\n📁 结果保存到: {filename} (共{len(results_df)}行，{len(results_df.columns)}列)")
            
            # 🔥 你的预览，显示T列
            print("\n📋 结果预览:")
            print(results_df.round(3)[['T', 'nb_groups', 'CP_cost', 'Total_cost', 'total_clustering_cpu_s']].head(10))
            
            successful_count += 1
            print(f"✅ Seed={seed} 完美！进度：{successful_count}/{TARGET_SEEDS}")
        else:
            print(f"❌ Seed={seed} 超时中断，尝试下一个...")
            
    except Exception as e:
        print(f"💥 Seed={seed} 异常：{str(e)}")
        stop_event.set()
        timeout_thread.join(timeout=1)


print(f"\n🎉 批量实验完成！成功{successful_count}/{TARGET_SEEDS}个seed")




🚀 Seed超时保护版 - 固定P=100，T=30-360 - 目标30个完整结果

🔄 Processing seed 1... (总超时3min)

🔄 Running T=30 with seed=1...
💥 Seed=1 异常：name 'PARAMS' is not defined

🔄 Processing seed 2... (总超时3min)

🔄 Running T=30 with seed=2...
💥 Seed=2 异常：name 'PARAMS' is not defined

🔄 Processing seed 3... (总超时3min)

🔄 Running T=30 with seed=3...
💥 Seed=3 异常：name 'PARAMS' is not defined

🔄 Processing seed 4... (总超时3min)

🔄 Running T=30 with seed=4...
💥 Seed=4 异常：name 'PARAMS' is not defined

🔄 Processing seed 5... (总超时3min)

🔄 Running T=30 with seed=5...
💥 Seed=5 异常：name 'PARAMS' is not defined

🔄 Processing seed 6... (总超时3min)

🔄 Running T=30 with seed=6...
💥 Seed=6 异常：name 'PARAMS' is not defined

🔄 Processing seed 7... (总超时3min)

🔄 Running T=30 with seed=7...
💥 Seed=7 异常：name 'PARAMS' is not defined

🔄 Processing seed 8... (总超时3min)

🔄 Running T=30 with seed=8...
💥 Seed=8 异常：name 'PARAMS' is not defined

🔄 Processing seed 9... (总超时3min)

🔄 Running T=30 with seed=9...
💥 Seed=9 异常：name 'PARAMS' is not defined

🔄 

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/Users/wenbo.wu/opt/anaconda3/envs/rte/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/Users/wenbo.wu/opt/anaconda3/envs/rte/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/Users/wenbo.wu/opt/anaconda3/envs/rte/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_13106/1515276769.py", line 980, in seed_timeout_monitor
__main__.SeedTimeoutException: Seed超时3分钟
Exception in thread Thread-5:
Traceback (most recent call last):
  File "/Users/wenbo.wu/opt/anaconda3/envs/rte/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/Users/wenbo.wu/opt/anaconda3/envs/rte/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/User

In [16]:
# plot sensitivity P for different seeds
import pandas as pd
import glob
import os

# 指定文件夹路径
folder_path = 'Sensitivity_P_COST'


# 查找所有匹配的CSV文件
pattern = os.path.join(folder_path, "Sensitivity_P10-200_seed*_all_metrics.csv")
csv_files = glob.glob(pattern)

if not csv_files:
    print("未找到匹配的CSV文件，请检查文件名模式。")
else:
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)

    # 垂直合并所有DataFrame（假设行数相同，对应行合并）
    combined_df = pd.concat(dfs, ignore_index=False)

    # 按行索引分组，计算每列均值（对应行平均）
    avg_df = combined_df.groupby(combined_df.index).mean()

    # 保存结果为average_results.csv，保留列名
    # avg_df.to_csv('average_results.csv')
    # print("平均结果已保存到 average_results.csv")
    print(avg_df)  # 预览前几行


        P  seed   nb_groups  Avg_tasks/group  CS_cost_before  CS_cost_after  \
0    10.0  15.5   49.866667         2.868633    14143.333333    4986.666667   
1    20.0  15.5   77.366667         3.739367    28770.000000    7736.666667   
2    30.0  15.5   93.800000         4.607167    42916.666667    9380.000000   
3    40.0  15.5  106.966667         5.396767    57703.333333   10696.666667   
4    50.0  15.5  112.566667         6.397333    71963.333333   11256.666667   
5    60.0  15.5  118.433333         7.299300    86363.333333   11843.333333   
6    70.0  15.5  124.766667         8.072533   100720.000000   12476.666667   
7    80.0  15.5  131.700000         8.727133   114913.333333   13170.000000   
8    90.0  15.5  137.900000         9.324133   128613.333333   13790.000000   
9   100.0  15.5  144.400000         9.884433   142690.000000   14440.000000   
10  110.0  15.5  151.200000        10.328733   156223.333333   15120.000000   
11  120.0  15.5  155.566667        11.029267   17154

In [17]:
# plot sensitivity T for different seeds
import pandas as pd
import glob
import os

# 指定文件夹路径
folder_path = 'Sensitivity_T_COST'


# 查找所有匹配的CSV文件
pattern = os.path.join(folder_path, "Sensitivity_T30-360_seed*_all_metrics.csv")
csv_files = glob.glob(pattern)

if not csv_files:
    print("未找到匹配的CSV文件，请检查文件名模式。")
else:
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)

    # 垂直合并所有DataFrame（假设行数相同，对应行合并）
    combined_df = pd.concat(dfs, ignore_index=False)

    # 按行索引分组，计算每列均值（对应行平均）
    avg_df = combined_df.groupby(combined_df.index).mean()

    # 保存结果为average_results.csv，保留列名
    # avg_df.to_csv('average_results.csv')
    # print("平均结果已保存到 average_results.csv")
    print(avg_df)  # 预览前几行


        T  seed   nb_groups  Avg_tasks/group  CS_cost_before  CS_cost_after  \
0    30.0  15.5   23.466667        10.258700    23953.333333    2346.666667   
1    60.0  15.5   47.266667        10.109800    47713.333333    4726.666667   
2    90.0  15.5   71.566667         9.984533    71383.333333    7156.666667   
3   120.0  15.5   96.233333         9.889667    95083.333333    9623.333333   
4   150.0  15.5  120.500000         9.871367   118883.333333   12050.000000   
5   180.0  15.5  144.400000         9.884433   142690.000000   14440.000000   
6   210.0  15.5  168.466667         9.860300   166130.000000   16846.666667   
7   240.0  15.5  192.433333         9.873367   190016.666667   19243.333333   
8   270.0  15.5  216.300000         9.884600   213843.333333   21630.000000   
9   300.0  15.5  240.700000         9.865867   237523.333333   24070.000000   
10  330.0  15.5  264.933333         9.853800   261136.666667   26493.333333   
11  360.0  15.5  289.066667         9.855033   28500